# 02 — Logistic Regression (Baseline Classifier)

**Status:** `PENDING MANUAL EXECUTION`


## 1. Imports and Configuration


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, brier_score_loss, log_loss
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
RANDOM_SEED = 42


## 2. Project Root & Paths


In [ ]:
def find_project_root() -> Path:
    cur = Path.cwd().resolve()
    for p in [cur] + list(cur.parents):
        if (p / "data").exists() and (p / "experiments").exists():
            return p
    raise RuntimeError("Could not locate project root.")

PROJECT_ROOT = find_project_root()
MODELS_DIR = PROJECT_ROOT / "models" / "phase_05"
REPORTS_DIR = PROJECT_ROOT / "reports" / "phase_05"


## 3. Load Dataset & Frozen Split Schema


In [ ]:
split_info = json.loads((MODELS_DIR / "phase05_split_and_schema.json").read_text(encoding="utf-8"))
df = pd.read_csv(PROJECT_ROOT / split_info["dataset"])
FEATURE_COLS = split_info["feature_cols"]


## 4. Feature & Target Separation


In [ ]:
X = df[FEATURE_COLS].copy()
y = (df["eligibility"].astype(str).str.upper() == "ELIGIBLE").astype(int)
train_idx = np.array(split_info["train_indices"])
X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]


## 5. Duplicate-Group & Split Validation


In [ ]:
groups = pd.util.hash_pandas_object(X_train, index=False).astype(str)
print("Train rows:", len(X_train), "| Unique groups:", groups.nunique())


## 6. Preprocessing Pipeline


In [ ]:
num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = [c for c in FEATURE_COLS if c not in num_cols]
preprocessor = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), num_cols),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="constant", fill_value="UNRECORDED")), ("ohe", OneHotEncoder(handle_unknown="ignore"))]), cat_cols),
])


## 7. Logistic Regression & 10-Fold StratifiedGroupKFold Configuration


In [ ]:
CV = StratifiedGroupKFold(n_splits=10, shuffle=True, random_state=RANDOM_SEED)
lr_pipe = Pipeline([("prep", preprocessor), ("clf", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_SEED))])


## 8. 10-Fold Cross-Validation & Calibration Evaluation (Manual Execution)


In [ ]:
print("Ready for 10-fold CV execution and Brier/LogLoss/ECE evaluation.")
